# 01 — Bronze Ingestion

Reads raw Parquet files from disk and loads them into the Bronze layer in MongoDB (tlc_bronze). Metadata tags (_meta) are injected at this stage. The ControlManager records the full execution lifecycle.

## Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, write_mongo, read_parquet
from src.paths import list_raw_files, str_path
from core.config.settings import settings
from core.audit.control_manager import ControlManager, ExecutionStatus
import pyspark.sql.functions as F
import uuid

VEHICLE_TYPES = ["yellow", "green", "fhv", "hvfhv"]
YEARS         = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

print("Configuration loaded.")

## Start Spark Session

In [ ]:
spark = get_spark(app_name="TLC_Bronze_Ingestion")
print(f"Spark version: {spark.version}")

## Run Bronze Ingestion Pipeline

In [ ]:
control = ControlManager("bronze_ingestion")
execution = control.start({"vehicle_types": VEHICLE_TYPES, "years": YEARS})

total_written = 0

for vehicle_type in VEHICLE_TYPES:
    files = list_raw_files(vehicle_type, years=YEARS)
    if not files:
        print(f"[WARN] No files found for vehicle_type='{vehicle_type}'. Skipping.")
        continue

    print(f"\n[{vehicle_type.upper()}] Processing {len(files)} file(s)...")

    for parquet_file in files:
        run_id = execution.execution_id
        print(f"  → {parquet_file.name}", end=" ")

        df = read_parquet(spark, str_path(parquet_file))

        # Inject metadata
        df = df.withColumn("_meta", F.struct(
            F.lit(vehicle_type).alias("vehicle_type"),
            F.lit(run_id).alias("run_id"),
            F.current_timestamp().alias("ingestion_time"),
            F.lit(parquet_file.name).alias("source_file"),
        ))

        n = write_mongo(df, settings.MONGO_DB_BRONZE, f"{vehicle_type}_raw", mode="append")
        total_written += n
        print(f"OK ({n:,} rows)")

control.end(
    ExecutionStatus.SUCCESS,
    {"total_records_written": total_written, "vehicle_types": VEHICLE_TYPES},
)
print(f"\nBronze ingestion complete. Total records: {total_written:,}")

## Audit Report

In [ ]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))